# 03_training_and_experiments.ipynb

This notebook trains an initial XGBoost model using the processed dataset created during the feature engineering workflow.

The notebook covers dataset preparation, data splitting, baseline model training, model evaluation, and experiment tracking. The resulting model serves as a baseline for future hyperparameter tuning and model optimization.

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [ ]:
import sys

sys.path.append("..")

## Import Dependencies

Import the required Python libraries and project modules.

In [ ]:
from datetime import datetime, timezone
import itertools
import json
from pathlib import Path

import time
import boto3
import joblib
import mlflow
import mlflow.xgboost
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    EVALUATION_REPORT_KEY,
    EVALUATION_REPORT_NAME,
    MLFLOW_TRACKING_SERVER_NAME,
    MLFLOW_TRACKING_SERVER_ARN,
    MLFLOW_EXPERIMENT_NAME,
    MODEL_ARTIFACT_KEY,
    MODEL_ARTIFACT_NAME,
    TARGET_COLUMN,
    TEST_DATA_KEY,
    TRAIN_DATA_KEY,
    VALIDATION_DATA_KEY,
)
from src.evaluate import evaluate_model
from src.train import train_baseline_model, split_features_target, connect_to_mlflow_tracking_server, track_baseline_experiment, track_hyperparameter_tuning

## Initialize AWS Clients

Initialize the AWS clients required for Amazon S3 and SageMaker.

In [ ]:
s3 = boto3.client("s3", region_name=AWS_REGION)
sm = boto3.client("sagemaker", region_name=AWS_REGION)

## Load Processed Dataset from Amazon S3

The processed dataset created during the feature engineering step is loaded from Amazon S3 for model training and evaluation.

In [ ]:
train_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=TRAIN_DATA_KEY,
)

train_df = pd.read_csv(
    train_obj["Body"]
)

validation_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=VALIDATION_DATA_KEY,
)

validation_df = pd.read_csv(
    validation_obj["Body"]
)

test_obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=TEST_DATA_KEY,
)

test_df = pd.read_csv(
    test_obj["Body"]
)

In [ ]:
print(f"Train dataset shape: {train_df.shape}")
print(f"Validation dataset shape: {validation_df.shape}")
print(f"Test dataset shape: {test_df.shape}")

## Inspect Processed Dataset

The processed dataset is inspected to verify its structure, feature representation, and overall readiness for model training.

This step confirms that the feature engineering workflow produced a clean dataset suitable for XGBoost.

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df["class"].value_counts(normalize=True)

### Dataset Summary

The processed dataset contains engineered numerical features and a binary target variable.

No additional preprocessing is required before splitting the dataset for model training.

## Prepare Training, Validation, and Test Sets

Separate features and target variables for model training, validation, and final evaluation.

In [ ]:
X_train, y_train = split_features_target(train_df, TARGET_COLUMN)
X_validation, y_validation = split_features_target(validation_df, TARGET_COLUMN)
X_test, y_test = split_features_target(test_df, TARGET_COLUMN)

## Train Baseline XGBoost Model

A baseline XGBoost classifier is trained using the training dataset.

The baseline model provides an initial performance benchmark and serves as a reference for future experiment tracking and hyperparameter tuning.

In [ ]:
model = train_baseline_model(X_train, y_train)

## Evaluate Baseline Model

The baseline model is evaluated on the validation dataset to assess its predictive performance.

Several classification metrics are calculated to establish a performance baseline before further model optimization.

In [ ]:
train_accuracy = model.score(X_train, y_train)

print(f"Train Accuracy: {train_accuracy:.4f}")

In [ ]:
results = evaluate_model(model, X_validation, y_validation)
validation_accuracy = results['accuracy']
validation_roc_auc = results['roc_auc']

print(f"Validation Accuracy: {validation_accuracy:.4f}")
print(f"Validation ROC AUC: {validation_roc_auc:.4f}")

print("\nClassification Report")
print(results["classification_report"])

### Baseline Model Results

The baseline XGBoost model achieved a validation accuracy of 66.7% and a ROC AUC score of 0.68.

The model shows signs of overfitting, with substantially higher performance on the training dataset than on the validation dataset.

These results serve as a baseline for future experiment tracking and hyperparameter tuning.

## Start MLflow Tracking Server

Start the SageMaker MLflow Tracking Server if it is currently stopped.

The tracking server is required for logging experiment runs, parameters, metrics, and model artifacts with MLflow. Starting the server only when needed helps reduce ongoing costs.

In [ ]:
tracking_server_response = sm.describe_mlflow_tracking_server(
    TrackingServerName=MLFLOW_TRACKING_SERVER_NAME,
)

tracking_server_status = tracking_server_response["TrackingServerStatus"]

print(f"Tracking server status: {tracking_server_status}")

if tracking_server_status in ["Stopped", "StopFailed"]:
    sm.start_mlflow_tracking_server(
        TrackingServerName=MLFLOW_TRACKING_SERVER_NAME,
    )

    print("Starting MLflow tracking server...")

    while True:
        tracking_server_response = sm.describe_mlflow_tracking_server(
            TrackingServerName=MLFLOW_TRACKING_SERVER_NAME,
        )

        tracking_server_status = tracking_server_response["TrackingServerStatus"]
        print(f"Current status: {tracking_server_status}")

        if tracking_server_status in ["Started", "Created"]:
            break

        if tracking_server_status in ["StartFailed", "CreateFailed"]:
            raise RuntimeError(
                f"MLflow tracking server failed: {tracking_server_status}"
            )

        time.sleep(60)

tracking_server_arn = tracking_server_response["TrackingServerArn"]

print(f"MLflow tracking server ARN: {tracking_server_arn}")

## Configure MLflow Tracking

Configure MLflow to use the SageMaker MLflow Tracking Server.

The experiment name groups all training and tuning runs for this project in one place.

In [ ]:
connect_to_mlflow_tracking_server(
    MLFLOW_TRACKING_SERVER_ARN,
    MLFLOW_TRACKING_SERVER_NAME,
)

## Track Baseline Experiment with MLflow

The baseline training run is logged to MLflow to capture model parameters, evaluation metrics, and model artifacts.

This enables experiment comparison, reproducibility, and model lifecycle tracking within SageMaker.

In [ ]:
track_baseline_experiment(
    model=model,
    validation_accuracy=validation_accuracy,
    validation_roc_auc=validation_roc_auc,
)

### Experiment Tracking Results

The baseline training run was successfully tracked using MLflow.

Model parameters, evaluation metrics, and model artifacts were stored in the SageMaker MLflow Tracking Server, enabling reproducibility and future comparison with additional training runs.

## Hyperparameter Tuning

A small grid search is used to evaluate multiple XGBoost configurations.

Each run is tracked in MLflow so that the validation performance of different parameter combinations can be compared directly.

In [ ]:
param_grid = {
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [50, 100],
    "subsample": [0.8, 1.0],
}

(
    best_model,
    best_params,
    best_results,
    best_auc,
    best_run_name,
) = track_hyperparameter_tuning(
    X_train,
    y_train,
    X_validation,
    y_validation,
    param_grid,
)

In [ ]:
print("Best run:", best_run_name)
print("Best params:", best_params)
print(f"Best validation ROC AUC: {best_auc:.4f}")

### Tuning Results

The grid search evaluated multiple XGBoost configurations and logged each run to MLflow.

The best model was selected based on validation ROC AUC, which is a suitable metric for credit risk classification.

## Final Model Evaluation

The best model identified during hyperparameter tuning is evaluated on the test dataset.

The test set was not used during model training or hyperparameter selection and therefore provides an unbiased estimate of the model's generalization performance.

In [ ]:
results = evaluate_model(best_model, X_test, y_test)

print(f"Test Accuracy: {results['accuracy']:.4f}")
print(f"Test ROC AUC: {results['roc_auc']:.4f}")

print("\nClassification Report")
print(results["classification_report"])

### Final Evaluation Results

The tuned XGBoost model achieved the best validation performance during hyperparameter tuning and was subsequently evaluated on the test dataset.

The resulting test metrics provide the final performance estimate and will be used in later stages of the machine learning lifecycle, including model registration, deployment, and pipeline automation.

## Upload Evaluation Results to Amazon S3

Store the final test evaluation results locally and upload them to Amazon S3.

The evaluation report is stored as a current report and as a versioned report under a timestamped S3 prefix so that it can be referenced later in the Model Registry.

In [ ]:
Path("data/evaluation").mkdir(
    parents=True,
    exist_ok=True,
)

local_evaluation_report_path = "data/evaluation/evaluation.json"

evaluation_report = {
    "accuracy": results["accuracy"],
    "precision": results["precision"],
    "recall": results["recall"],
    "f1": results["f1"],
    "roc_auc": results["roc_auc"],
}

with open(local_evaluation_report_path, "w") as f:
    json.dump(evaluation_report, f, indent=2)

s3.upload_file(
    local_evaluation_report_path,
    BUCKET_NAME,
    EVALUATION_REPORT_KEY,
)

evaluation_report_s3_uri = f"s3://{BUCKET_NAME}/{EVALUATION_REPORT_KEY}"

print(f"Evaluation report uploaded to {evaluation_report_s3_uri}")

## Upload Model Artifact to Amazon S3

Save the best trained model locally and upload it to Amazon S3.

The uploaded model artifact is used by the next notebook to create a SageMaker-compatible model package.

In [ ]:
Path("data/models").mkdir(
    parents=True,
    exist_ok=True,
)

local_model_path = "data/models/best_model.pkl"

joblib.dump(
    best_model,
    local_model_path,
)

s3.upload_file(
    local_model_path,
    BUCKET_NAME,
    MODEL_ARTIFACT_KEY,
)

model_artifact_s3_uri = f"s3://{BUCKET_NAME}/{MODEL_ARTIFACT_KEY}"

print(f"Model artifact uploaded to {model_artifact_s3_uri}")

## Stop MLflow Tracking Server

Stop the SageMaker MLflow Tracking Server after the experiment tracking workflow is complete.

Stopping the tracking server helps avoid ongoing costs while keeping the logged MLflow experiments and artifacts available for later use.

In [ ]:
tracking_server_response = sm.describe_mlflow_tracking_server(
    TrackingServerName=MLFLOW_TRACKING_SERVER_NAME,
)

tracking_server_status = tracking_server_response["TrackingServerStatus"]

print(f"Tracking server status: {tracking_server_status}")

if tracking_server_status != "Stopped":
    sm.stop_mlflow_tracking_server(
        TrackingServerName=MLFLOW_TRACKING_SERVER_NAME,
    )

    print(f"Stopping MLflow tracking server: {MLFLOW_TRACKING_SERVER_NAME}")

while True:
    tracking_server_response = sm.describe_mlflow_tracking_server(
        TrackingServerName=MLFLOW_TRACKING_SERVER_NAME,
    )

    tracking_server_status = tracking_server_response["TrackingServerStatus"]

    print(f"Current status: {tracking_server_status}")

    if tracking_server_status == "Stopped":
        break

    if tracking_server_status == "StopFailed":
        raise RuntimeError(
            f"MLflow tracking server failed to stop: {tracking_server_status}"
        )

    time.sleep(60)

print("MLflow tracking server stopped.")

## Result

The best performing XGBoost model and its evaluation results have been uploaded to Amazon S3.

The uploaded artifacts include the trained model artifact and the evaluation report. The model artifact will be used to create a SageMaker-compatible model package, while the evaluation report provides the final test metrics for later model registration.

These artifacts will be used in subsequent stages of the machine learning lifecycle, including SageMaker model creation, explainability analysis with Clarify, model registration, deployment, and pipeline automation.